In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1YV5ZEQWw8ka1BLNAIz7-RhR7LzVzh1kx", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_01_intro.mp3"))

# 🚀 CLAUDE.md Hierarchy & Rules — Hands-On Configuration Lab

Welcome to your first hands-on lab for Domain 3 of the Claude Certified Architect exam. In this notebook, we will **simulate** the entire CLAUDE.md configuration system using Python — building a working model of how Claude Code loads, merges, and prioritizes configuration files.

By the end of this notebook, you will be able to:
- Explain the three levels of CLAUDE.md (user, project, directory)
- Predict which rules win when levels conflict
- Use `@import` to organize modular configuration
- Create path-specific rules with glob patterns in `.claude/rules/`
- Debug configuration issues using the mental model you build here

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/claude-code-workflows/practice/1/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you join a team that uses Claude Code. Your first prompt generates code that is *functional* — but uses the wrong framework, the wrong test library, and the wrong file structure. Your teammate types the same prompt and gets perfect output.

The difference? **Configuration.** Your teammate's project has a carefully structured set of CLAUDE.md files that shape Claude's behavior at every level. Understanding this hierarchy is worth 20% of your exam score.

Let us build a simulation of this system from scratch.

In [ ]:
#@title 🎧 Listen: Setup Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_setup_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Setup — we'll simulate Claude Code's configuration system in pure Python
import os
import json
import fnmatch
import tempfile
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, field

# Create a temporary workspace to simulate a real project
WORKSPACE = tempfile.mkdtemp(prefix="claude_config_lab_")
print(f"Workspace created at: {WORKSPACE}")

In [ ]:
#@title 🎧 Listen: Three Levels
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_three_levels.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — The Three-Level Hierarchy

Think of CLAUDE.md configuration like layers of clothing:

- **User-level** (`~/.claude/CLAUDE.md`) — Your underwear. Personal preferences that go everywhere with you. Not shared with anyone.
- **Project-level** (`.claude/CLAUDE.md` or root `CLAUDE.md`) — Your team uniform. Shared via git. Everyone on the team wears the same one.
- **Directory-level** (`subdirectory/CLAUDE.md`) — A specialized apron for specific work areas. The kitchen staff wears one, the servers do not.

The critical rule: **most specific wins**. A directory-level rule overrides a project-level rule, which overrides a user-level rule. Let us model this.

In [ ]:
#@title 🎧 Listen: Parser Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_parser_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class ConfigRule:
    """A single configuration rule with its source level."""
    key: str           # e.g., "indentation", "test_framework"
    value: str         # e.g., "2 spaces", "vitest"
    level: str         # "user", "project", or "directory"
    source_file: str   # path to the CLAUDE.md that defined this rule

    def __repr__(self):
        return f"Rule({self.key}={self.value!r}, level={self.level})"


@dataclass
class ClaudeMdFile:
    """Represents a parsed CLAUDE.md file."""
    path: str
    level: str  # "user", "project", or "directory"
    rules: List[ConfigRule] = field(default_factory=list)
    imports: List[str] = field(default_factory=list)


def parse_claude_md(content: str, path: str, level: str) -> ClaudeMdFile:
    """Parse a CLAUDE.md file content into structured rules."""
    config = ClaudeMdFile(path=path, level=level)

    for line in content.strip().split("\n"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        # Handle @import directives
        if line.startswith("@import"):
            import_path = line.replace("@import", "").strip()
            config.imports.append(import_path)
            continue

        # Parse key: value rules
        if ":" in line:
            key, value = line.split(":", 1)
            config.rules.append(ConfigRule(
                key=key.strip().lower(),
                value=value.strip(),
                level=level,
                source_file=path
            ))

    return config


# Let's test with a simple example
sample_content = """# Project Conventions
framework: Next.js 14
test_framework: Vitest
indentation: 2 spaces
"""

parsed = parse_claude_md(sample_content, ".claude/CLAUDE.md", "project")
print("Parsed rules:")
for rule in parsed.rules:
    print(f"  {rule}")

In [ ]:
#@title 🎧 Listen: Think About This
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_think_about_this.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 🤔 Think About This

Before we implement the merge logic, ask yourself:
- If user-level says `indentation: tabs` and project-level says `indentation: 2 spaces`, which wins?
- What if a directory-level file says nothing about indentation — does the project rule still apply?
- Why would you ever want user-level rules if project rules always override them?

The answers: project wins over user (more specific), yes the project rule applies (rules merge, they don't replace), and user-level rules cover things the project doesn't specify (like your preferred explanation verbosity).

In [ ]:
#@title 🎧 Listen: Merge Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_merge_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Priority values — higher number = higher priority
LEVEL_PRIORITY = {
    "user": 1,      # Lowest priority — personal defaults
    "project": 2,   # Middle — team conventions
    "directory": 3,  # Highest — directory-specific overrides
}


def merge_configs(configs: List[ClaudeMdFile]) -> Dict[str, ConfigRule]:
    """
    Merge multiple CLAUDE.md files with precedence rules.
    Most specific (directory) wins over less specific (user).
    """
    merged = {}

    # Sort by priority (lowest first, so higher priority overwrites)
    sorted_configs = sorted(configs, key=lambda c: LEVEL_PRIORITY[c.level])

    for config in sorted_configs:
        for rule in config.rules:
            # Later (higher priority) rules overwrite earlier ones
            merged[rule.key] = rule

    return merged


# Test the merge with conflicting rules
user_config = parse_claude_md(
    "indentation: tabs\ncomments: verbose\ntest_framework: jest",
    "~/.claude/CLAUDE.md", "user"
)

project_config = parse_claude_md(
    "indentation: 2 spaces\ntest_framework: vitest\nframework: nextjs",
    ".claude/CLAUDE.md", "project"
)

directory_config = parse_claude_md(
    "indentation: 4 spaces\nerror_handling: withErrorHandler middleware",
    "src/api/CLAUDE.md", "directory"
)

merged = merge_configs([user_config, project_config, directory_config])

print("Merged configuration (most specific wins):")
print("-" * 55)
for key, rule in sorted(merged.items()):
    print(f"  {key:20s} = {rule.value:25s}  (from {rule.level})")

Notice what happened:
- **indentation** was defined at all three levels. Directory-level (4 spaces) won.
- **test_framework** was defined at user (jest) and project (vitest). Project won.
- **comments** was only defined at user-level, so it survived untouched.
- **framework** was only at project-level — it passes through.
- **error_handling** was only at directory-level — it adds without conflict.

This is exactly what we want. The merge is additive (all rules combine) but with override semantics (most specific wins for conflicts).

In [ ]:
#@title 🎧 Listen: Import System
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_import_system.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The @import System

As projects grow, a single CLAUDE.md becomes unwieldy. The `@import` directive lets you split rules into focused files.

Let us implement this.

In [ ]:
#@title 🎧 Listen: Imports Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_imports_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def create_project_structure(base_dir: str):
    """Create a realistic project with modular CLAUDE.md configuration."""

    # Main CLAUDE.md with @import directives
    main_claude_md = """# Project Instructions
@import ./rules/testing.md
@import ./rules/api-conventions.md
@import ./rules/database.md

framework: Next.js 14
language: TypeScript strict
"""

    # Imported rule files
    testing_rules = """# Testing Conventions
test_framework: Vitest
test_location: co-located with source
coverage_minimum: 80%
mock_style: vi.mock()
"""

    api_rules = """# API Conventions
api_style: REST with versioning
response_format: { data: T, error: null } | { data: null, error: string }
middleware: withErrorHandler required
validation: Zod schemas
"""

    database_rules = """# Database Conventions
orm: Prisma
migration_tool: prisma migrate
naming: snake_case for columns
"""

    # Create the directory structure
    os.makedirs(os.path.join(base_dir, ".claude", "rules"), exist_ok=True)

    files = {
        ".claude/CLAUDE.md": main_claude_md,
        ".claude/rules/testing.md": testing_rules,
        ".claude/rules/api-conventions.md": api_rules,
        ".claude/rules/database.md": database_rules,
    }

    for rel_path, content in files.items():
        full_path = os.path.join(base_dir, rel_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, "w") as f:
            f.write(content)

    return files


def resolve_imports(config: ClaudeMdFile, base_dir: str) -> List[ClaudeMdFile]:
    """Resolve @import directives by reading and parsing imported files."""
    all_configs = [config]

    for import_path in config.imports:
        # Resolve relative to the CLAUDE.md file's directory
        claude_md_dir = os.path.dirname(os.path.join(base_dir, config.path))
        full_path = os.path.normpath(os.path.join(claude_md_dir, import_path))

        if os.path.exists(full_path):
            with open(full_path) as f:
                imported = parse_claude_md(f.read(), import_path, config.level)
                all_configs.append(imported)
        else:
            print(f"  ⚠️ Import not found: {import_path}")

    return all_configs


# Create the project and resolve imports
files = create_project_structure(WORKSPACE)
print("Created project structure:")
for path in sorted(files.keys()):
    print(f"  📄 {path}")

# Parse the main CLAUDE.md
with open(os.path.join(WORKSPACE, ".claude/CLAUDE.md")) as f:
    main_config = parse_claude_md(f.read(), ".claude/CLAUDE.md", "project")

print(f"\nMain config has {len(main_config.imports)} imports:")
for imp in main_config.imports:
    print(f"  📥 {imp}")

# Resolve all imports
all_configs = resolve_imports(main_config, WORKSPACE)

print(f"\nResolved to {len(all_configs)} config files:")
for c in all_configs:
    print(f"  {c.path}: {len(c.rules)} rules")

# Merge everything
merged = merge_configs(all_configs)
print(f"\nFinal merged configuration ({len(merged)} rules):")
print("-" * 55)
for key, rule in sorted(merged.items()):
    print(f"  {key:20s} = {rule.value}")

In [ ]:
#@title 🎧 Listen: Path Rules
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_path_rules.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Path-Specific Rules with .claude/rules/

Now let us implement the most powerful feature: rules that activate **only when editing matching files**. This uses YAML frontmatter with glob patterns.

In [ ]:
#@title 🎧 Listen: Path Rules Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_path_rules_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class PathRule:
    """A rule that only applies to files matching specific glob patterns."""
    patterns: List[str]    # Glob patterns like "**/*.test.tsx"
    rules: List[ConfigRule]
    source_file: str

    def matches(self, file_path: str) -> bool:
        """Check if a file path matches any of this rule's patterns."""
        for pattern in self.patterns:
            if fnmatch.fnmatch(file_path, pattern):
                return True
        return False


def parse_path_rule(content: str, source_file: str) -> Optional[PathRule]:
    """Parse a .claude/rules/ file with YAML frontmatter."""
    lines = content.strip().split("\n")

    # Check for YAML frontmatter
    if not lines or lines[0].strip() != "---":
        return None

    # Find the closing ---
    frontmatter_end = None
    for i in range(1, len(lines)):
        if lines[i].strip() == "---":
            frontmatter_end = i
            break

    if frontmatter_end is None:
        return None

    # Extract paths from frontmatter
    patterns = []
    for line in lines[1:frontmatter_end]:
        line = line.strip()
        if line.startswith("- "):
            # Remove quotes and the dash prefix
            pattern = line[2:].strip().strip('"').strip("'")
            patterns.append(pattern)

    # Parse rules from the body (after frontmatter)
    body = "\n".join(lines[frontmatter_end + 1:])
    config = parse_claude_md(body, source_file, "path-specific")

    return PathRule(
        patterns=patterns,
        rules=config.rules,
        source_file=source_file
    )


# Create path-specific rule files
path_rules_content = {
    ".claude/rules/testing.md": """---
paths:
  - "**/*.test.tsx"
  - "**/*.test.ts"
  - "**/*.spec.ts"
---
# Testing Conventions
test_style: use describe/it blocks
mock_approach: vi.mock() for external services
assertion_lib: built-in Vitest expect
snapshot_required: yes for UI components
""",

    ".claude/rules/terraform.md": """---
paths:
  - "terraform/**/*"
  - "infra/**/*.tf"
---
# Terraform Conventions
formatting: terraform fmt style
tagging: environment, team, managed-by required
state_backend: S3 with DynamoDB locking
ami_source: data sources only, never hardcoded
""",

    ".claude/rules/api-routes.md": """---
paths:
  - "src/api/**/*.ts"
  - "src/routes/**/*.ts"
---
# API Route Conventions
middleware: withErrorHandler required
response_format: { data, error } envelope
validation: Zod schema on all inputs
auth: requireAuth middleware on protected routes
""",
}

# Write the files
for rel_path, content in path_rules_content.items():
    full_path = os.path.join(WORKSPACE, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w") as f:
        f.write(content)

# Parse all path rules
parsed_path_rules = []
for rel_path, content in path_rules_content.items():
    rule = parse_path_rule(content, rel_path)
    if rule:
        parsed_path_rules.append(rule)
        print(f"📋 {rel_path}")
        print(f"   Patterns: {rule.patterns}")
        print(f"   Rules: {len(rule.rules)}")
        print()

Now let us see path-specific rules in action. We will simulate Claude editing different files and check which rules activate.

In [ ]:
#@title 🎧 Listen: Activation Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_activation_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def get_active_rules(file_path: str, path_rules: List[PathRule]) -> List[ConfigRule]:
    """Get all rules that apply to a specific file being edited."""
    active = []
    for pr in path_rules:
        if pr.matches(file_path):
            active.extend(pr.rules)
    return active


# Simulate editing different files
test_files = [
    "src/components/Button.test.tsx",
    "src/components/Button.tsx",
    "src/api/v2/users.ts",
    "terraform/modules/vpc/main.tf",
    "README.md",
]

print("Path-Specific Rule Activation")
print("=" * 65)

for file_path in test_files:
    active = get_active_rules(file_path, parsed_path_rules)
    if active:
        print(f"\n📝 Editing: {file_path}")
        print(f"   Active rules ({len(active)}):")
        for rule in active:
            print(f"     • {rule.key}: {rule.value}")
    else:
        print(f"\n📝 Editing: {file_path}")
        print(f"   No path-specific rules activated (only base CLAUDE.md applies)")

This is exactly what we want. Test rules activate for test files. Terraform rules activate for `.tf` files. API rules activate for API routes. And `README.md` gets only the base configuration.

---

### ✋ Stop and Think

Before scrolling down, answer these questions:
1. If you have both a directory-level CLAUDE.md in `src/api/` AND a path-specific rule matching `src/api/**/*.ts`, which one takes effect?
2. Can a single file match multiple path-specific rules?

*Answers: Both take effect — they merge together. Yes, a file can match multiple rules (e.g., `src/api/users.test.ts` matches both the testing and API rules).*

In [ ]:
#@title 🎧 Listen: Todo Engine
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_todo_engine.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. 🔧 Your Turn — Build a Complete Configuration System

Now let us put it all together. Complete the `ConfigurationEngine` class that simulates how Claude Code loads all configuration for a given file.

In [ ]:
class ConfigurationEngine:
    """
    Simulates Claude Code's full configuration loading system.

    Loads:
    1. User-level CLAUDE.md
    2. Project-level CLAUDE.md (with @import resolution)
    3. Directory-level CLAUDE.md files (walking up from the file's directory)
    4. Path-specific rules from .claude/rules/
    """

    def __init__(self, project_root: str, user_home: str = None):
        self.project_root = project_root
        self.user_home = user_home or os.path.expanduser("~")
        self.path_rules: List[PathRule] = []
        self.base_configs: List[ClaudeMdFile] = []

    def load_all(self):
        """Load all configuration sources."""
        # Load user-level
        user_path = os.path.join(self.user_home, ".claude", "CLAUDE.md")
        if os.path.exists(user_path):
            with open(user_path) as f:
                config = parse_claude_md(f.read(), user_path, "user")
                self.base_configs.extend(resolve_imports(config, self.user_home))

        # Load project-level
        for candidate in [".claude/CLAUDE.md", "CLAUDE.md"]:
            proj_path = os.path.join(self.project_root, candidate)
            if os.path.exists(proj_path):
                with open(proj_path) as f:
                    config = parse_claude_md(f.read(), candidate, "project")
                    self.base_configs.extend(resolve_imports(config, self.project_root))
                break

        # Load path-specific rules from .claude/rules/
        rules_dir = os.path.join(self.project_root, ".claude", "rules")
        if os.path.isdir(rules_dir):
            for fname in sorted(os.listdir(rules_dir)):
                if fname.endswith(".md"):
                    with open(os.path.join(rules_dir, fname)) as f:
                        rule = parse_path_rule(f.read(), f".claude/rules/{fname}")
                        if rule:
                            self.path_rules.append(rule)

    def get_config_for_file(self, file_path: str) -> Dict[str, ConfigRule]:
        """
        Get the merged configuration for a specific file being edited.

        Merge order (lowest to highest priority):
        1. User-level rules
        2. Project-level rules (including imports)
        3. Path-specific rules that match this file
        4. Directory-level CLAUDE.md (if any)
        """
        # ============ TODO ============
        # Step 1: Start with base configs (user + project)
        # Step 2: Find and add path-specific rules that match file_path
        # Step 3: Check for directory-level CLAUDE.md
        # Step 4: Merge everything with proper precedence
        # Hint: Create ConfigRule objects for path-specific matches
        #        with level="path-specific" and priority between
        #        project (2) and directory (3).
        # ==============================

        all_configs = list(self.base_configs)  # Start with base

        # Add path-specific rules that match
        for pr in self.path_rules:
            if pr.matches(file_path):
                # Wrap path-specific rules as a ClaudeMdFile for merging
                path_config = ClaudeMdFile(
                    path=pr.source_file,
                    level="project",  # Same priority as project
                    rules=pr.rules
                )
                all_configs.append(path_config)

        # Check for directory-level CLAUDE.md
        file_dir = os.path.dirname(file_path)
        dir_claude_md = os.path.join(self.project_root, file_dir, "CLAUDE.md")
        if os.path.exists(dir_claude_md):
            with open(dir_claude_md) as f:
                dir_config = parse_claude_md(f.read(), f"{file_dir}/CLAUDE.md", "directory")
                all_configs.append(dir_config)

        return merge_configs(all_configs)

    def debug_config(self, file_path: str):
        """Print a debug view showing where each rule comes from."""
        config = self.get_config_for_file(file_path)
        print(f"\n🔍 Configuration for: {file_path}")
        print("=" * 60)
        for key, rule in sorted(config.items()):
            print(f"  {key:22s} = {rule.value:25s}  ← {rule.level} ({rule.source_file})")
        return config

In [ ]:
#@title 🎧 Listen: Engine Test Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_engine_test_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Create a user-level CLAUDE.md for our simulation
user_claude_dir = os.path.join(WORKSPACE, "_user_home", ".claude")
os.makedirs(user_claude_dir, exist_ok=True)
with open(os.path.join(user_claude_dir, "CLAUDE.md"), "w") as f:
    f.write("""# Personal Preferences
explanation_verbosity: detailed
indentation: tabs
comments: always add JSDoc
""")

# Create a directory-level CLAUDE.md for the API directory
api_dir = os.path.join(WORKSPACE, "src", "api")
os.makedirs(api_dir, exist_ok=True)
with open(os.path.join(api_dir, "CLAUDE.md"), "w") as f:
    f.write("""# API Directory Rules
error_handling: always use withErrorHandler
rate_limiting: required on all public endpoints
indentation: 4 spaces
""")

# Initialize and load the engine
engine = ConfigurationEngine(
    project_root=WORKSPACE,
    user_home=os.path.join(WORKSPACE, "_user_home")
)
engine.load_all()

print(f"Loaded {len(engine.base_configs)} base configs, {len(engine.path_rules)} path rules")

# Test with different files
engine.debug_config("src/components/Button.tsx")
engine.debug_config("src/components/Button.test.tsx")
engine.debug_config("src/api/v2/users.ts")
engine.debug_config("terraform/modules/vpc/main.tf")

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. 📊 Visualization — Configuration Precedence Map

Let us create a visual representation of which rules activate for different file types.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Define file types and which config sources apply
file_types = [
    "README.md",
    "src/utils.ts",
    "src/Button.test.tsx",
    "src/api/users.ts",
    "terraform/main.tf",
]

config_sources = ["User-level", "Project-level", "@import rules", "Path-specific", "Directory-level"]
colors = ["#3498db", "#2ecc71", "#9b59b6", "#e67e22", "#e74c3c"]

# Which sources apply to each file type (1 = yes, 0 = no)
activation_matrix = [
    [1, 1, 1, 0, 0],  # README.md
    [1, 1, 1, 0, 0],  # src/utils.ts
    [1, 1, 1, 1, 0],  # src/Button.test.tsx (path-specific testing rules)
    [1, 1, 1, 1, 1],  # src/api/users.ts (path-specific + directory)
    [1, 1, 1, 1, 0],  # terraform/main.tf (path-specific terraform rules)
]

bar_height = 0.6
for i, file_type in enumerate(file_types):
    x_offset = 0
    for j, (source, color) in enumerate(zip(config_sources, colors)):
        if activation_matrix[i][j]:
            ax.barh(i, 1, left=x_offset, height=bar_height, color=color,
                    edgecolor='white', linewidth=1)
            x_offset += 1

ax.set_yticks(range(len(file_types)))
ax.set_yticklabels(file_types, fontsize=11, family='monospace')
ax.set_xlabel("Configuration Sources Active", fontsize=12)
ax.set_title("Which Configuration Sources Apply to Each File Type", fontsize=14, fontweight='bold')

# Legend
patches = [mpatches.Patch(color=c, label=s) for s, c in zip(config_sources, colors)]
ax.legend(handles=patches, loc='lower right', fontsize=10)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig("config_precedence.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Configuration precedence map generated!")

In [ ]:
#@title 🎧 Listen: Todo Exam
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_todo_exam.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. 🔧 Your Turn — Exam Practice Questions

Test your understanding with these exam-style scenarios.

In [ ]:
# ============ TODO ============
# For each scenario, predict what value Claude will use.
# Fill in your answers, then run the verification cell below.

# Scenario 1: User says "tabs", Project says "2 spaces". What does Claude use?
scenario_1_answer = "???"  # YOUR ANSWER HERE

# Scenario 2: Project CLAUDE.md says "test_framework: jest" but
# .claude/rules/testing.md (path-specific) says "test_framework: vitest".
# You're editing Button.test.tsx. What framework does Claude use?
scenario_2_answer = "???"  # YOUR ANSWER HERE

# Scenario 3: You have a directory-level CLAUDE.md in src/api/ that says
# "validation: manual". The project-level says "validation: zod".
# You're editing src/api/users.ts. Which wins?
scenario_3_answer = "???"  # YOUR ANSWER HERE

# Scenario 4: You add a personal rule to ~/.claude/CLAUDE.md that says
# "always_explain: true". Will your teammates see this rule?
scenario_4_answer = "???"  # YOUR ANSWER HERE (yes/no)

# ==============================

In [ ]:
# ✅ Verification
answers = {
    "scenario_1": ("2 spaces", scenario_1_answer.lower().strip()),
    "scenario_2": ("vitest", scenario_2_answer.lower().strip()),
    "scenario_3": ("manual", scenario_3_answer.lower().strip()),
    "scenario_4": ("no", scenario_4_answer.lower().strip()),
}

explanations = {
    "scenario_1": "Project-level (more specific) overrides user-level.",
    "scenario_2": "Path-specific rules match the test file pattern and apply alongside project rules. Since both define test_framework, the last-loaded (path-specific) wins.",
    "scenario_3": "Directory-level is most specific — it overrides project-level.",
    "scenario_4": "User-level CLAUDE.md is NOT version controlled. It stays on your machine.",
}

correct = 0
for key, (expected, given) in answers.items():
    if expected in given or given in expected:
        print(f"✅ {key}: Correct! {explanations[key]}")
        correct += 1
    else:
        print(f"❌ {key}: Expected '{expected}', got '{given}'. {explanations[key]}")

print(f"\nScore: {correct}/4")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Reflection and Next Steps

**Key takeaways:**
1. CLAUDE.md has three levels: user → project → directory. Most specific wins.
2. User-level is personal and NOT shared via version control.
3. `@import` keeps configuration modular and reviewable.
4. Path-specific rules in `.claude/rules/` use glob patterns to target file types across directories.
5. Use `/memory` to debug what Claude actually sees.

**What to try next:**
- Create a `.claude/rules/` file for your own project's test conventions
- Split a large CLAUDE.md into imported modules
- Experiment with directory-level CLAUDE.md for different parts of a monorepo

In [ ]:
# Cleanup
shutil.rmtree(WORKSPACE)
print("✅ Workspace cleaned up. Notebook complete!")